In [1]:
!git clone https://github.com/cuhksz-nlp/R2Gen.git
%cd R2Gen

!pip install kagglehub
import kagglehub
path = kagglehub.dataset_download("raddar/chest-xrays-indiana-university")
print(path)

fatal: destination path 'R2Gen' already exists and is not an empty directory.
/home/jupyter-st126489/NLU/Project/Progress/Ablation/R2Gen/R2Gen


/opt/tljh/user/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Defaulting to user installation because normal site-packages is not writeable
/home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2


In [2]:
import os
from pathlib import Path

root = Path(path)   # 'path' is from kagglehub.dataset_download(...)

print("Dataset root:", root)
print("\nTop-level files/folders:")
for p in sorted(root.iterdir()):
    print("-", p.name)

print("\nSample tree (2 levels):")
for p in sorted(root.rglob("*")):
    rel = p.relative_to(root)
    if len(rel.parts) <= 2:
        print(rel)

Dataset root: /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2

Top-level files/folders:
- images
- indiana_projections.csv
- indiana_reports.csv

Sample tree (2 levels):
images
images/images_normalized
indiana_projections.csv
indiana_reports.csv


In [3]:
import pandas as pd

png_files = list(root.rglob("*.png"))
print("PNG files:", len(png_files))


PNG files: 7470


In [4]:
from pathlib import Path
import shutil

project_root = Path("R2GenCMN")
iu_root = project_root / "data" / "iu_xray"
images_dst = iu_root / "images"

images_dst.mkdir(parents=True, exist_ok=True)

print("Created:", iu_root)
print("Created:", images_dst)

Created: R2GenCMN/data/iu_xray
Created: R2GenCMN/data/iu_xray/images


In [5]:
from pathlib import Path
import shutil

src_exts = {".png", ".jpg", ".jpeg"}
image_files = [p for p in root.rglob("*") if p.suffix.lower() in src_exts]

print("Found images:", len(image_files))

copied = 0
for img in image_files:
    dst = images_dst / img.name
    if not dst.exists():
        shutil.copy2(img, dst)
        copied += 1

print("Copied images:", copied)
print("Total now in destination:", len(list(images_dst.glob("*"))))

Found images: 7470
Copied images: 0
Total now in destination: 7470


In [6]:
print("Example destination files:")
for i, p in enumerate(sorted(images_dst.iterdir())):
    print(p.name)
    if i == 9:
        break

Example destination files:
1000_IM-0003-1001.dcm.png
1000_IM-0003-2001.dcm.png
1000_IM-0003-3001.dcm.png
1001_IM-0004-1001.dcm.png
1001_IM-0004-1002.dcm.png
1002_IM-0004-1001.dcm.png
1002_IM-0004-2001.dcm.png
1003_IM-0005-2002.dcm.png
1004_IM-0005-1001.dcm.png
1004_IM-0005-2001.dcm.png


In [7]:
import pandas as pd

csv_files = list(root.rglob("*.csv"))
print("CSV files found:")
for i, f in enumerate(csv_files):
    print(i, f)

# Try reading each CSV briefly
for f in csv_files:
    print("\n---", f.name, "---")
    try:
        df = pd.read_csv(f)
        print("shape:", df.shape)
        print("columns:", list(df.columns))
        print(df.head(2))
    except Exception as e:
        print("Could not read:", e)

CSV files found:
0 /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/indiana_reports.csv
1 /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/indiana_projections.csv

--- indiana_reports.csv ---
shape: (3851, 8)
columns: ['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']
   uid                                               MeSH  \
0    1                                             normal   
1    2  Cardiomegaly/borderline;Pulmonary Artery/enlarged   

                        Problems                                image  \
0                         normal            Xray Chest PA and Lateral   
1  Cardiomegaly;Pulmonary Artery  Chest, 2 views, frontal and lateral   

                 indication comparison  \
0          Positive TB test      None.   
1  Preop bariatric surgery.      None.   

                                            findings  \
0  The

In [8]:
report_csv = None

for f in csv_files:
    if "report" in f.name.lower() or "indiana" in f.name.lower():
        report_csv = f
        break

print("Selected CSV:", report_csv)

df = pd.read_csv(report_csv)
print(df.columns.tolist())
print(df.head())

Selected CSV: /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/indiana_reports.csv
['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']
   uid                                               MeSH  \
0    1                                             normal   
1    2  Cardiomegaly/borderline;Pulmonary Artery/enlarged   
2    3                                             normal   
3    4  Pulmonary Disease, Chronic Obstructive;Bullous...   
4    5  Osteophyte/thoracic vertebrae/multiple/small;T...   

                                            Problems  \
0                                             normal   
1                      Cardiomegaly;Pulmonary Artery   
2                                             normal   
3  Pulmonary Disease, Chronic Obstructive;Bullous...   
4                         Osteophyte;Thickening;Lung   

                                               image  \
0                

In [9]:
import pandas as pd
from pathlib import Path

csv_files = list(Path(path).rglob("*.csv"))

print("CSV files found:")
for i, f in enumerate(csv_files):
    print(i, f)

# Pick the report CSV manually after seeing the list if needed
report_csv = csv_files[0]
df = pd.read_csv(report_csv)

print("\nSelected CSV:", report_csv)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nHead:")
display(df.head())

CSV files found:
0 /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/indiana_reports.csv
1 /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/indiana_projections.csv

Selected CSV: /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/indiana_reports.csv
Shape: (3851, 8)
Columns: ['uid', 'MeSH', 'Problems', 'image', 'indication', 'comparison', 'findings', 'impression']

Head:


,uid,MeSH,Problems,image,indication,comparison,findings,impression
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p..."
3,4,"Pulmonary Disease, Chronic Obstructive;Bullous...","Pulmonary Disease, Chronic Obstructive;Bullous...","PA and lateral views of the chest XXXX, XXXX a...",XXXX-year-old XXXX with XXXX.,None available,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,Osteophyte/thoracic vertebrae/multiple/small;T...,Osteophyte;Thickening;Lung,Xray Chest PA and Lateral,Chest and nasal congestion.,NaN,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.


In [10]:
# Check missing values in likely important columns
candidate_cols = [c for c in df.columns if c.lower() in ["uid", "id", "findings", "impression", "image", "filename", "problems"]]
print("Candidate columns:", candidate_cols)

for c in candidate_cols:
    print(f"{c}: missing =", df[c].isna().sum())

Candidate columns: ['uid', 'Problems', 'image', 'findings', 'impression']
uid: missing = 0
Problems: missing = 0
image: missing = 0
findings: missing = 514
impression: missing = 31


In [11]:
import pandas as pd
from pathlib import Path

csv_files = list(Path(path).rglob("*.csv"))

print("CSV files found:")
for i, f in enumerate(csv_files):
    print(i, f)

# choose the correct CSV index if needed
report_csv = csv_files[0]
df = pd.read_csv(report_csv)

print("\nSelected CSV:", report_csv)
print("\nExact columns:")
for i, c in enumerate(df.columns):
    print(i, repr(c))

CSV files found:
0 /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/indiana_reports.csv
1 /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/indiana_projections.csv

Selected CSV: /home/jupyter-st126489/.cache/kagglehub/datasets/raddar/chest-xrays-indiana-university/versions/2/indiana_reports.csv

Exact columns:
0 'uid'
1 'MeSH'
2 'Problems'
3 'image'
4 'indication'
5 'comparison'
6 'findings'
7 'impression'


In [12]:
import pandas as pd
from pathlib import Path

reports_csv = Path("../data/indiana_reports.csv")
projections_csv = Path("../data/indiana_projections.csv")

reports_df = pd.read_csv(reports_csv)
proj_df = pd.read_csv(projections_csv)

reports_df.columns = [c.strip() for c in reports_df.columns]
proj_df.columns = [c.strip() for c in proj_df.columns]

print("Reports columns:")
for i, c in enumerate(reports_df.columns):
    print(i, repr(c))

print("\nProjections columns:")
for i, c in enumerate(proj_df.columns):
    print(i, repr(c))

print("\nReports head:")
display(reports_df.head())

print("\nProjections head:")
display(proj_df.head())

Reports columns:
0 'uid'
1 'MeSH'
2 'Problems'
3 'image'
4 'indication'
5 'comparison'
6 'findings'
7 'impression'

Projections columns:
0 'uid'
1 'filename'
2 'projection'

Reports head:


,uid,MeSH,Problems,image,indication,comparison,findings,impression
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p..."
3,4,"Pulmonary Disease, Chronic Obstructive;Bullous...","Pulmonary Disease, Chronic Obstructive;Bullous...","PA and lateral views of the chest XXXX, XXXX a...",XXXX-year-old XXXX with XXXX.,None available,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...
4,5,Osteophyte/thoracic vertebrae/multiple/small;T...,Osteophyte;Thickening;Lung,Xray Chest PA and Lateral,Chest and nasal congestion.,NaN,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.



Projections head:


,uid,filename,projection
0,1,1_IM-0001-4001.dcm.png,Frontal
1,1,1_IM-0001-3001.dcm.png,Lateral
2,2,2_IM-0652-1001.dcm.png,Frontal
3,2,2_IM-0652-2001.dcm.png,Lateral
4,3,3_IM-1384-1001.dcm.png,Frontal


In [13]:
lower_map_reports = {c.lower(): c for c in reports_df.columns}

print(lower_map_reports)

ID_COL = lower_map_reports.get("uid")
TARGET_COL = lower_map_reports.get("impression")

print("ID_COL =", ID_COL)
print("TARGET_COL =", TARGET_COL)

{'uid': 'uid', 'mesh': 'MeSH', 'problems': 'Problems', 'image': 'image', 'indication': 'indication', 'comparison': 'comparison', 'findings': 'findings', 'impression': 'impression'}
ID_COL = uid
TARGET_COL = impression


In [14]:
reports_clean = reports_df.copy()

reports_clean = reports_clean[reports_clean[TARGET_COL].notna()]
reports_clean[TARGET_COL] = reports_clean[TARGET_COL].astype(str).str.strip()
reports_clean = reports_clean[reports_clean[TARGET_COL] != ""]

print("Rows with valid impression:", len(reports_clean))
display(reports_clean[[ID_COL, TARGET_COL]].head())

Rows with valid impression: 3820


,uid,impression
0,1,Normal chest x-XXXX.
1,2,No acute pulmonary findings.
2,3,"No displaced rib fractures, pneumothorax, or p..."
3,4,1. Bullous emphysema and interstitial fibrosis...
4,5,No acute cardiopulmonary abnormality.


In [15]:
 proj_df["filename"] = proj_df["filename"].astype(str).str.strip()

uid_to_images = proj_df.groupby("uid")["filename"].apply(list).to_dict()

print("Number of uid entries with images:", len(uid_to_images))

sample_items = list(uid_to_images.items())[:5]
for uid, imgs in sample_items:
    print(uid, imgs)

Number of uid entries with images: 3851
1 ['1_IM-0001-4001.dcm.png', '1_IM-0001-3001.dcm.png']
2 ['2_IM-0652-1001.dcm.png', '2_IM-0652-2001.dcm.png']
3 ['3_IM-1384-1001.dcm.png', '3_IM-1384-2001.dcm.png']
4 ['4_IM-2050-1001.dcm.png', '4_IM-2050-2001.dcm.png']
5 ['5_IM-2117-1003002.dcm.png', '5_IM-2117-1004003.dcm.png']


In [16]:
reports_clean["image_filenames"] = reports_clean[ID_COL].map(uid_to_images)

merged_df = reports_clean.copy()
merged_df = merged_df[merged_df["image_filenames"].notna()]
merged_df = merged_df[merged_df["image_filenames"].apply(lambda x: isinstance(x, list) and len(x) > 0)]

print("Rows after merging with images:", len(merged_df))
display(merged_df[[ID_COL, TARGET_COL, "image_filenames"]].head())

Rows after merging with images: 3820


,uid,impression,image_filenames
0,1,Normal chest x-XXXX.,"[1_IM-0001-4001.dcm.png, 1_IM-0001-3001.dcm.png]"
1,2,No acute pulmonary findings.,"[2_IM-0652-1001.dcm.png, 2_IM-0652-2001.dcm.png]"
2,3,"No displaced rib fractures, pneumothorax, or p...","[3_IM-1384-1001.dcm.png, 3_IM-1384-2001.dcm.png]"
3,4,1. Bullous emphysema and interstitial fibrosis...,"[4_IM-2050-1001.dcm.png, 4_IM-2050-2001.dcm.png]"
4,5,No acute cardiopulmonary abnormality.,"[5_IM-2117-1003002.dcm.png, 5_IM-2117-1004003...."


In [17]:
from pathlib import Path

images_dst = Path("R2GenCMN/data/iu_xray/images")
existing_images = set(p.name for p in images_dst.iterdir() if p.is_file())

def keep_existing(img_list):
    return [img for img in img_list if img in existing_images]

merged_df["image_filenames"] = merged_df["image_filenames"].apply(keep_existing)
merged_df = merged_df[merged_df["image_filenames"].apply(len) > 0]

print("Rows after checking actual image files:", len(merged_df))
display(merged_df[[ID_COL, TARGET_COL, "image_filenames"]].head())

Rows after checking actual image files: 3820


,uid,impression,image_filenames
0,1,Normal chest x-XXXX.,"[1_IM-0001-4001.dcm.png, 1_IM-0001-3001.dcm.png]"
1,2,No acute pulmonary findings.,"[2_IM-0652-1001.dcm.png, 2_IM-0652-2001.dcm.png]"
2,3,"No displaced rib fractures, pneumothorax, or p...","[3_IM-1384-1001.dcm.png, 3_IM-1384-2001.dcm.png]"
3,4,1. Bullous emphysema and interstitial fibrosis...,"[4_IM-2050-1001.dcm.png, 4_IM-2050-2001.dcm.png]"
4,5,No acute cardiopulmonary abnormality.,"[5_IM-2117-1003002.dcm.png, 5_IM-2117-1004003...."


In [18]:
from sklearn.model_selection import train_test_split

case_df = merged_df[[ID_COL, TARGET_COL, "image_filenames", "Problems"]].copy()
case_df = case_df.drop_duplicates(subset=[ID_COL]).reset_index(drop=True)

train_df, temp_df = train_test_split(case_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=2/3, random_state=42)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 2674
Val: 382
Test: 764


In [19]:
def make_examples(df_split):
    examples = []
    for _, row in df_split.iterrows():

        # 🔥 Extract Problems column
        problems = ""
        if "Problems" in row and not pd.isna(row["Problems"]):
            problems = str(row["Problems"])

        examples.append({
            "id": str(row[ID_COL]),
            "image_path": row["image_filenames"],
            "report": str(row[TARGET_COL]).strip(),
            "Problems": problems   # 🔥 ADD THIS LINE
        })

    return examples

annotation = {
    "train": make_examples(train_df),
    "val": make_examples(val_df),
    "test": make_examples(test_df),
}

print("Train example:")
print(annotation["train"][0])

Train example:
{'id': '1240', 'image_path': ['1240_IM-0162-4001.dcm.png', '1240_IM-0162-2001.dcm.png'], 'report': '1. Stable left lung consolidation, possibly infectious pneumonia and/or aspiration. Recommend followup radiographs after treatment to ensure complete resolution. 2. Stable mild residual medial right basilar airspace disease.', 'Problems': 'Tube, Inserted;Catheters, Indwelling;Consolidation;Airspace Disease;Atherosclerosis;Spine'}


In [20]:
import json
from pathlib import Path

ann_path = Path("../data/iu_xray/annotation.json")
ann_path.parent.mkdir(parents=True, exist_ok=True)

with open(ann_path, "w", encoding="utf-8") as f:
    json.dump(annotation, f, indent=2, ensure_ascii=False)

print("Saved annotation to:", ann_path)

Saved annotation to: ../data/iu_xray/annotation.json


In [21]:
with open("../data/iu_xray/annotation.json", "r", encoding="utf-8") as f:
    ann = json.load(f)

print("Keys:", ann.keys())
print("Train size:", len(ann["train"]))
print("Val size:", len(ann["val"]))
print("Test size:", len(ann["test"]))
print("\nSample entry:")
print(ann["train"][0])

Keys: dict_keys(['train', 'val', 'test'])
Train size: 2674
Val size: 382
Test size: 764

Sample entry:
{'id': '1240', 'image_path': ['1240_IM-0162-4001.dcm.png', '1240_IM-0162-2001.dcm.png'], 'report': '1. Stable left lung consolidation, possibly infectious pneumonia and/or aspiration. Recommend followup radiographs after treatment to ensure complete resolution. 2. Stable mild residual medial right basilar airspace disease.', 'Problems': 'Tube, Inserted;Catheters, Indwelling;Consolidation;Airspace Disease;Atherosclerosis;Spine'}


In [22]:
!grep -R "image_path\|report\|ann_path\|split\|json.load\|tokenizer" -n ../modules ../data ../main.py

../modules/datasets.py:12:    def __init__(self, args, tokenizer, split, transform=None):
../modules/datasets.py:15:        self.ann_path = args.ann_path
../modules/datasets.py:17:        self.split = split
../modules/datasets.py:18:        self.tokenizer = tokenizer
../modules/datasets.py:22:        with open(self.ann_path, 'r') as f:
../modules/datasets.py:23:            self.ann = json.load(f)
../modules/datasets.py:25:        # 🔥 Build label vocabulary from ALL splits
../modules/datasets.py:28:        # Current split data
../modules/datasets.py:29:        self.examples = self.ann[self.split]
../modules/datasets.py:31:        # Tokenize reports
../modules/datasets.py:33:            tokens = tokenizer(self.examples[i]['report'])
../modules/datasets.py:46:        for split in ann:  # train / val / test
../modules/datasets.py:47:            for sample in ann[split]:
../modules/datasets.py:50:                    items = [p.strip() for p in probs.split(',')]
../modules/datasets.py:63:   

In [23]:
for file in [
    "../modules/dataloaders.py",
    "../modules/datasets.py",
    "../modules/tokenizers.py",
]:
    print(f"\n===== {file} =====")
    try:
        with open(file, "r", encoding="utf-8") as f:
            lines = f.readlines()
        for i, line in enumerate(lines[:220], start=1):
            print(f"{i:4d}: {line.rstrip()}")
    except Exception as e:
        print("Could not read:", e)


===== ../modules/dataloaders.py =====
   1: import torch
   2: import numpy as np
   3: from torchvision import transforms
   4: from torch.utils.data import DataLoader
   5: from .datasets import IuxrayMultiImageDataset, MimiccxrSingleImageDataset
   6: 
   7: 
   8: class R2DataLoader(DataLoader):
   9:     def __init__(self, args, tokenizer, split, shuffle):
  10:         self.args = args
  11:         self.dataset_name = args.dataset_name
  12:         self.batch_size = args.batch_size
  13:         self.shuffle = shuffle
  14:         self.num_workers = args.num_workers
  15:         self.tokenizer = tokenizer
  16:         self.split = split
  17: 
  18:         if split == 'train':
  19:             self.transform = transforms.Compose([
  20:                 transforms.Resize(256),
  21:                 transforms.RandomCrop(224),
  22:                 transforms.RandomHorizontalFlip(),
  23:                 transforms.ToTensor(),
  24:                 transforms.Normalize((0.4

In [25]:
import sys
#sys.path.append("../R2Gen")

from argparse import Namespace
from modules.tokenizers import Tokenizer
from modules.dataloaders import R2DataLoader

args = Namespace(
    image_dir="R2GenCMN/data/iu_xray/images",
    ann_path="../data/iu_xray/annotation.json",
    dataset_name="iu_xray",
    max_seq_length=60,
    threshold=3,
    num_workers=2,
    batch_size=2,
)

print("Building tokenizer...")
tokenizer = Tokenizer(args)
print("Tokenizer vocab size:", len(tokenizer.token2idx))

print("Building train loader...")
train_loader = R2DataLoader(args, tokenizer, split="train", shuffle=True)

batch = next(iter(train_loader))
print("Loaded one batch successfully.")
print(type(batch))
print(batch)

Building tokenizer...
Tokenizer vocab size: 669
Building train loader...
Loaded one batch successfully.
<class 'tuple'>
(('682', '1584'), tensor([[[[[-1.1418, -1.0904, -1.0562,  ..., -1.8439, -2.0323, -2.0665],
           [-0.8849, -0.8507, -0.8335,  ..., -1.5185, -1.7412, -1.9980],
           [-0.7137, -0.7137, -0.7137,  ..., -1.3644, -1.4843, -1.6898],
           ...,
           [-2.0665, -2.0665, -2.0665,  ..., -0.7479, -0.7137, -0.6623],
           [-2.0665, -2.0665, -2.0665,  ..., -0.7308, -0.7137, -0.6623],
           [-2.0665, -2.0665, -2.0665,  ..., -0.7308, -0.7137, -0.6623]],

          [[-1.0378, -0.9853, -0.9503,  ..., -1.7556, -1.9482, -1.9832],
           [-0.7752, -0.7402, -0.7227,  ..., -1.4230, -1.6506, -1.9132],
           [-0.6001, -0.6001, -0.6001,  ..., -1.2654, -1.3880, -1.5980],
           ...,
           [-1.9832, -1.9832, -1.9832,  ..., -0.6352, -0.6001, -0.5476],
           [-1.9832, -1.9832, -1.9832,  ..., -0.6176, -0.6001, -0.5476],
           [-1.9832, -1.9

In [26]:
file_path = "../modules/datasets.py"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i, line in enumerate(lines, start=1):
    if "class " in line or "__init__" in line or "__getitem__" in line or "image_path" in line or "report" in line or "ann_path" in line:
        start = max(1, i - 8)
        end = min(len(lines), i + 25)
        print(f"\n--- lines {start} to {end} ---")
        for j in range(start, end + 1):
            print(f"{j:4d}: {lines[j-1].rstrip()}")


--- lines 3 to 36 ---
   3: import torch
   4: import numpy as np
   5: from PIL import Image, ImageFile
   6: from torch.utils.data import Dataset
   7: 
   8: ImageFile.LOAD_TRUNCATED_IMAGES = True
   9: 
  10: 
  11: class BaseDataset(Dataset):
  12:     def __init__(self, args, tokenizer, split, transform=None):
  13: 
  14:         self.image_dir = args.image_dir
  15:         self.ann_path = args.ann_path
  16:         self.max_seq_length = args.max_seq_length
  17:         self.split = split
  18:         self.tokenizer = tokenizer
  19:         self.transform = transform
  20: 
  21:         # Load annotations
  22:         with open(self.ann_path, 'r') as f:
  23:             self.ann = json.load(f)
  24: 
  25:         # 🔥 Build label vocabulary from ALL splits
  26:         self.label2idx = self.build_label_vocab(self.ann)
  27: 
  28:         # Current split data
  29:         self.examples = self.ann[self.split]
  30: 
  31:         # Tokenize reports
  32:         for i 

In [27]:
file_path = "../modules/datasets.py"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

keywords = ["class BaseDataset", "def __init__", "json.load", "self.examples", "tokenizer", "ids", "mask", "report"]

for i, line in enumerate(lines, start=1):
    if any(k in line for k in keywords):
        start = max(1, i - 10)
        end = min(len(lines), i + 35)
        print(f"\n--- lines {start} to {end} ---")
        for j in range(start, end + 1):
            print(f"{j:4d}: {lines[j-1].rstrip()}")


--- lines 1 to 46 ---
   1: import os
   2: import json
   3: import torch
   4: import numpy as np
   5: from PIL import Image, ImageFile
   6: from torch.utils.data import Dataset
   7: 
   8: ImageFile.LOAD_TRUNCATED_IMAGES = True
   9: 
  10: 
  11: class BaseDataset(Dataset):
  12:     def __init__(self, args, tokenizer, split, transform=None):
  13: 
  14:         self.image_dir = args.image_dir
  15:         self.ann_path = args.ann_path
  16:         self.max_seq_length = args.max_seq_length
  17:         self.split = split
  18:         self.tokenizer = tokenizer
  19:         self.transform = transform
  20: 
  21:         # Load annotations
  22:         with open(self.ann_path, 'r') as f:
  23:             self.ann = json.load(f)
  24: 
  25:         # 🔥 Build label vocabulary from ALL splits
  26:         self.label2idx = self.build_label_vocab(self.ann)
  27: 
  28:         # Current split data
  29:         self.examples = self.ann[self.split]
  30: 
  31:         # Tok

In [28]:
import sys
#sys.path.append("R2Gen")

from argparse import Namespace
from modules.tokenizers import Tokenizer
from modules.dataloaders import R2DataLoader

args = Namespace(
    image_dir="R2GenCMN/data/iu_xray/images",
    ann_path="../data/iu_xray/annotation.json",
    dataset_name="iu_xray",
    max_seq_length=60,
    threshold=3,
    num_workers=2,
    batch_size=2,
)

print("Building tokenizer...")
tokenizer = Tokenizer(args)
print("Tokenizer built.")
print("Vocab size:", len(tokenizer.token2idx))

print("\nBuilding train loader...")
train_loader = R2DataLoader(args, tokenizer, split="train", shuffle=True)

print("Fetching one batch...")
batch = next(iter(train_loader))

print("Batch loaded successfully.")
print("Batch type:", type(batch))
print("Batch length:", len(batch))

Building tokenizer...
Tokenizer built.
Vocab size: 669

Building train loader...
Fetching one batch...
Batch loaded successfully.
Batch type: <class 'tuple'>
Batch length: 4


In [29]:
for i, item in enumerate(batch):
    print(f"\nItem {i}:")
    if hasattr(item, "shape"):
        print("shape:", item.shape)
    else:
        print(type(item))
        try:
            print(item[:2])
        except:
            print(item)


Item 0:
<class 'tuple'>
('3652', '734')

Item 1:
shape: torch.Size([2, 2, 3, 224, 224])

Item 2:
shape: torch.Size([2, 8])

Item 3:
shape: torch.Size([2, 8])


In [30]:
!cat ../run_iu_xray.sh

python main.py \
  --image_dir R2GenCMN/data/iu_xray/images \
  --ann_path data/iu_xray/annotation.json \
  --dataset_name iu_xray \
  --max_seq_length 40 \
  --threshold 3 \
  --batch_size 8 \
  --epochs 5 \
  --save_dir results/iu_xray_test \
  --step_size 50 \
  --gamma 0.1 \
  --seed 9223 \
  --log_period 50 \
  --problem_dim 100 \
  


In [31]:
import json
import numpy as np
from pathlib import Path

ann_path = Path("../data/iu_xray/annotation.json")

with open(ann_path, "r", encoding="utf-8") as f:
    ann = json.load(f)

all_reports = []
for split in ["train", "val", "test"]:
    all_reports.extend([x["report"] for x in ann[split]])

lengths = [len(str(r).split()) for r in all_reports]

print("Num reports:", len(lengths))
print("Min:", np.min(lengths))
print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("90th percentile:", np.percentile(lengths, 90))
print("95th percentile:", np.percentile(lengths, 95))
print("99th percentile:", np.percentile(lengths, 99))
print("Max:", np.max(lengths))

Num reports: 3820
Min: 1
Mean: 10.562303664921465
Median: 5.0
90th percentile: 25.0
95th percentile: 35.04999999999973
99th percentile: 56.809999999999945
Max: 130


In [32]:
from collections import Counter

img_counts = merged_df["image_filenames"].apply(len)
print(img_counts.value_counts().sort_index())

image_filenames
1     435
2    3191
3     180
4      13
5       1
Name: count, dtype: int64


In [33]:
bad_rows = merged_df[merged_df["image_filenames"].apply(len) < 2]
print("Bad rows:", len(bad_rows))
display(bad_rows[[ID_COL, TARGET_COL, "image_filenames"]].head(20))

Bad rows: 435


,uid,impression,image_filenames
43,44,No acute cardiopulmonary disease.,[44_IM-2078-1001.dcm.png]
44,45,Stable cardiomegaly without overt pulmonary ed...,[45_IM-2081-1001.dcm.png]
59,60,1. Round area of density measuring 1.9 x 1.8 c...,[60_IM-2192-1001.dcm.png]
67,68,Rib films. No fractures or dislocations. Chest...,[68_IM-2251-1001.dcm.png]
70,71,No acute cardiopulmonary disease.,[71_IM-2273-1001.dcm.png]
72,73,Improved basilar aeration. Persistent small bi...,[73_IM-2289-1001.dcm.png]
73,74,No acute pulmonary disease. Multiple thoracic ...,[74_IM-2296-2001.dcm.png]
80,81,No acute cardiopulmonary abnormality identified.,[81_IM-2343-2001.dcm.png]
87,88,1. Findings consistent with mild congestive he...,[88_IM-2394-2001.dcm.png]
90,91,Moderate sized right pneumothorax. There is mi...,[91_IM-2415-2001.dcm.png]


In [34]:
from sklearn.model_selection import train_test_split
import json
from pathlib import Path
import pandas as pd

# ✅ KEEP Problems column
case_df = merged_df[[ID_COL, TARGET_COL, "image_filenames", "Problems"]].copy()

case_df = case_df.drop_duplicates(subset=[ID_COL]).reset_index(drop=True)

# keep only rows with >= 2 images
case_df = case_df[
    case_df["image_filenames"].apply(lambda x: isinstance(x, list) and len(x) >= 2)
].copy()

print("Cases with >=2 images:", len(case_df))

# keep first two images only
case_df["image_filenames"] = case_df["image_filenames"].apply(lambda x: x[:2])

# sanity check
print(case_df["image_filenames"].apply(len).value_counts())


# ================= SPLIT =================
train_df, temp_df = train_test_split(case_df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=2/3, random_state=42)


# ================= BUILD JSON =================
def make_examples(df_split):
    examples = []
    for _, row in df_split.iterrows():

        # ✅ extract Problems safely
        problems = ""
        if "Problems" in row and not pd.isna(row["Problems"]):
            problems = str(row["Problems"]).strip()

        examples.append({
            "id": str(row[ID_COL]),
            "image_path": row["image_filenames"],
            "report": str(row[TARGET_COL]).strip(),
            "Problems": problems   # 🔥 IMPORTANT
        })

    return examples


annotation = {
    "train": make_examples(train_df),
    "val": make_examples(val_df),
    "test": make_examples(test_df),
}


# ================= SAVE =================
ann_path = Path("../data/iu_xray/annotation.json")

with open(ann_path, "w", encoding="utf-8") as f:
    json.dump(annotation, f, indent=2, ensure_ascii=False)

print("Saved:", ann_path)
print("Train:", len(annotation["train"]))
print("Val:", len(annotation["val"]))
print("Test:", len(annotation["test"]))

# 🔍 CHECK
print("Sample:", annotation["train"][0])

Cases with >=2 images: 3385
image_filenames
2    3385
Name: count, dtype: int64
Saved: ../data/iu_xray/annotation.json
Train: 2369
Val: 338
Test: 678
Sample: {'id': '499', 'image_path': ['499_IM-2116-1001.dcm.png', '499_IM-2116-2001.dcm.png'], 'report': 'No acute cardiopulmonary abnormality.', 'Problems': 'normal'}


In [35]:
import json

with open("../data/iu_xray/annotation.json", "r", encoding="utf-8") as f:
    ann = json.load(f)

for split in ["train", "val", "test"]:
    bad = [x for x in ann[split] if len(x["image_path"]) < 2]
    print(split, "bad samples:", len(bad))

train bad samples: 0
val bad samples: 0
test bad samples: 0


In [36]:
from pathlib import Path

file_path = Path("../modules/trainer.py")
text = file_path.read_text(encoding="utf-8")

text = text.replace(
    "record_table = record_table.append(self.best_record['val'], ignore_index=True)",
    "record_table = pd.concat([record_table, pd.DataFrame([self.best_record['val']])], ignore_index=True)"
)

text = text.replace(
    "record_table = record_table.append(self.best_record['test'], ignore_index=True)",
    "record_table = pd.concat([record_table, pd.DataFrame([self.best_record['test']])], ignore_index=True)"
)

file_path.write_text(text, encoding="utf-8")
print("Patched trainer.py")

Patched trainer.py


In [37]:
!grep -n "record_table = pd.concat\|append(" ../modules/trainer.py

119:        record_table = pd.concat([


In [38]:
#%cd R2Gen

!python main.py \
  --image_dir R2GenCMN/data/iu_xray/images \
  --ann_path ../data/iu_xray/annotation.json \
  --dataset_name iu_xray \
  --max_seq_length 40 \
  --threshold 3 \
  --batch_size 8 \
  --epochs 5 \
  --save_dir results/iu_xray_test \
  --step_size 50 \
  --gamma 0.1 \
  --seed 9223

/home/jupyter-st126489/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jupyter-st126489/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Traceback (most recent call last):
  File "/home/jupyter-st126489/NLU/Project/Progress/Ablation/R2Gen/R2Gen/main.py", line 124, in <module>
    main()
  File "/home/jupyter-st126489/NLU/Project/Progress/Ablation/R2Gen/R2Gen/main.py", line 120, in main
    trainer.train()
  File "/home/jupyter-st126489/NLU/Project/Progress/Ablation/R2Ge